In [30]:
import pandas as pd
import datetime as dt
from acled import Acled
import numpy as np

In [2]:
acled = Acled()

INFO:acled:Access token correctly retrieved.


Train: Data from January 2018 to December 2022. This includes the 2018 Sudanese revolution but excludes the 2023 Civil War.
Test onset civil war: Data from January 2023 to December 2023 which includes the escalation of the civil war.
Test active civil war: Data from January 2024 to December 2025 which includes fluctuations in ongoing civil war.


In [3]:
countries = ["Sudan"]
start_date = "2017-07-01"
end_date = "2024-12-31"

train_start_date = "2018-01-01"
train_end_date = "2022-12-31"

onset_start_date = "2023-01-01"
onset_end_date ="2023-12-31"

active_start_date = "2024-01-01"
active_end_date ="2024-12-31"

In [36]:
all_data = acled.get_data(countries, start_date, end_date)

INFO:acled:Requesting data...
INFO:acled:Requesting data...
INFO:acled:Requesting data...
INFO:acled:Requesting data...
INFO:acled:Requesting data...
INFO:acled:All data successfully fetched.


In [6]:
def mark_conflict_events(df: pd.DataFrame) -> pd.DataFrame:
    # 1 = Conflict event (Y)
    # 0 = Non-conflict (used for features)

    acled_subevent_mapping = {
        # BATTLES (Conflict)
        "Armed clash": 1,
        "Government regains territory": 1,
        "Non-state actor overtakes territory": 1,

        # EXPLOSIONS / REMOTE VIOLENCE (Conflict)
        "Air/drone strike": 1,
        "Chemical weapon": 1,
        "Remote explosive/landmine/IED": 1,
        "Shelling/artillery/missile attack": 1,
        "Suicide bomb": 1,
        "Grenade": 1,

        # VIOLENCE AGAINST CIVILIANS (Conflict)
        "Abduction/forced disappearance": 1,
        "Attack": 1,
        "Sexual violence": 1,

        # RIOTS (Conflict)
        "Mob violence": 1,
        "Violent demonstration": 1,

        # PROTESTS (Non-conflict)
        "Excessive force against protesters": 0,
        "Peaceful protest": 0,
        "Protest with intervention": 0,

        # STRATEGIC DEVELOPMENTS (Non-conflict)
        "Agreement": 0,
        "Arrests": 0,
        "Change to group/activity": 0,
        "Disrupted weapons use": 0,
        "Headquarters or base established": 0,
        "Looting/property destruction": 0,
        "Non-violent transfer of territory": 0,
        "Other": 0
    }
    df["conflict"] = df["sub_event_type"].apply(lambda x: acled_subevent_mapping[x])
    return df


In [31]:
def create_regional_monthly_baseline(df):
    df = df.copy()

    df_grouped = df.groupby(["admin2", "year_month"])["conflict"].sum().reset_index(name="conflict_event_count") # Sun only conflict events
    df_grouped = df_grouped.sort_values(by=['admin2', 'year_month'])

    df_grouped['rolling_mean_6m'] = df_grouped.groupby('admin2')['conflict_event_count'].transform(
        lambda x: x.rolling(window=6, min_periods=6).mean().shift(1)
    )

    df_grouped['rolling_std_6m'] = df_grouped.groupby('admin2')['conflict_event_count'].transform(
        lambda x: x.rolling(window=6, min_periods=6).std().shift(1)
    )

    k = 1.0 # TODO find the best k
    df_grouped['escalation_threshold'] = df_grouped['rolling_mean_6m'] + (k * df_grouped['rolling_std_6m'])

    df_grouped['target_escalation'] = np.where(
        df_grouped['conflict_event_count'] > df_grouped['escalation_threshold'], 1, 0
    )

    return df_grouped

In [32]:
def pre_process_data(df):
    df = mark_conflict_events(all_data)

    pivot_df = pd.pivot_table(
        df,
        values="event_id_cnty",
        index=["admin2", "year_month"],
        columns=["sub_event_type"],
        aggfunc="count",
        fill_value=0
    ).reset_index()

    # Change column names to be easier to process
    pivot_df.columns = (
        pivot_df.columns
        .str.lower()
        .str.replace(' ', '_', regex=False)
        .str.replace('/', '_', regex=False)
        .str.replace('-', '_', regex=False)
    )

    baseline_df = create_regional_monthly_baseline(df)

    fatalities_df = df.groupby(['admin2', 'year_month'])['fatalities'].sum().reset_index()

    combined_df = pd.merge(baseline_df, pivot_df, on=['admin2', 'year_month'], how='left')
    combined_df = pd.merge(combined_df, fatalities_df, on=['admin2', 'year_month'], how='left')

    event_cols = pivot_df.columns.drop(['admin2', 'year_month']).tolist()
    combined_df[event_cols] = combined_df[event_cols].fillna(0)
    combined_df['fatalities'] = combined_df['fatalities'].fillna(0)

    # Lag all predictors to create auto-regressive features
    predictor_cols = event_cols + ['fatalities']
    combined_df[predictor_cols] = combined_df.groupby('admin2')[predictor_cols].shift(1)
    combined_df = combined_df.rename(columns={"admin2": "region"})
    return combined_df, predictor_cols

In [44]:
def calculate_conflict_ratio(df):
    count_0 = (df["target_escalation"] == 0).sum()
    count_1 = (df["target_escalation"] == 1).sum()
    ratio = count_1 / count_0

    return {"non-escalation": count_0, "escalation": count_1, "ratio": ratio}

In [45]:
processed_df, X = pre_process_data(all_data)

train_df = processed_df[
    (processed_df['year_month'] >= train_start_date)
    &(processed_df['year_month'] <= train_end_date)
    ].copy()

ratios = calculate_conflict_ratio(train_df)
print(ratios)

{'non-escalation': np.int64(2291), 'escalation': np.int64(340), 'ratio': np.float64(0.14840680925360106)}


In [39]:

#
# onset_test_df = df_grouped[
#     (df_grouped['year_month'] >= onset_start_date) &
#     (df_grouped['year_month'] <= onset_end_date)
# ].copy()
#
# active_test_df = df_grouped[
#     (df_grouped['year_month'] >= active_start_date) &
#     (df_grouped['year_month'] <= active_end_date)
# ].copy()

In [40]:
train_df

,region,year_month,conflict_event_count,rolling_mean_6m,rolling_std_6m,escalation_threshold,target_escalation,abduction_forced_disappearance,agreement,air_drone_strike,...,non_state_actor_overtakes_territory,non_violent_transfer_of_territory,other,peaceful_protest,protest_with_intervention,remote_explosive_landmine_ied,sexual_violence,shelling_artillery_missile_attack,violent_demonstration,fatalities
0,Abassiya,2018-06,1,NaN,NaN,NaN,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Abassiya,2018-09,1,NaN,NaN,NaN,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Abassiya,2018-12,1,NaN,NaN,NaN,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,Abassiya,2019-04,0,NaN,NaN,NaN,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
4,Abassiya,2019-09,2,NaN,NaN,NaN,0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4505,Zalingi,2022-08,1,1.000000,0.894427,1.894427,0,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0
4506,Zalingi,2022-09,0,0.833333,0.752773,1.586106,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4507,Zalingi,2022-10,0,0.500000,0.547723,1.047723,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4508,Zalingi,2022-11,3,0.500000,0.547723,1.047723,1,0.0,0.0,0.0,...,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0
